# Task 3 — TinyResNet-18-PM E4 Experiments

This notebook trains only the parameter-matched E4 architecture. It does **not** retrain
E1, E2, or E3.

1. Gender starts from accepted E1 and changes only SmallCNN to TinyResNet-18-PM.
2. Usage starts from accepted E2, keeps its class-balanced loss, and changes only the architecture.

The residual model starts from random weights, stays near 395,000 parameters, and uses the same
80×60 images, folds, seed 2753, optimiser, schedule, and 30 epochs. Select a Colab GPU runtime,
then use Run All. If seed 2753 fails any frozen gate, stop; do not run confirmation seeds.


## 1. Mount Drive and load the submitted branch

Drive supplies the dataset, accepted E1/E2 parents, registry, and persistent E4 output.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

In [2]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip().splitlines()
    if dirty:
        print("Local repository changes found:")
        for change in dirty:
            print(f"  {change}")
        print("Trying a safe fast-forward update. Git will stop before overwriting a local file.")
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")

Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Branch: task-3-gender-usage-classification
Commit: 9c644e88e1b45e02bcb39e82b4de643c468b29ce


## 2. Copy the teacher data onto the runtime disk

Training reads images from Colab's local disk. The archive keeps the repository folder structure.


In [3]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

Extracting 44,441 teacher images...
Teacher data ready: 44,441 images


## 3. Resolve the accepted parents and verify TinyResNet E4

Gender resolves its five E1 folds. Usage resolves its five accepted E2 folds. The checks build
both TinyResNet heads, verify their output shapes, parameters, and MAC budgets, and take no
optimiser step.


In [4]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (DRIVE_TASK_DIR / "experiments", DRIVE_TASK_DIR / "logs", DRIVE_TASK_DIR / "results"):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
)

gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_parent_run_ids = latest_completed_usage_e2_parent_run_ids(
    output_root=DRIVE_TASK_DIR
)
gender_e4_check = check_task3_child_setup(
    "gender_tinyresnet18_pm",
    parent_run_ids=gender_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)
usage_e4_check = check_task3_child_setup(
    "usage_tinyresnet18_pm",
    parent_run_ids=usage_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)

for check in (gender_e4_check, usage_e4_check):
    if check["parameter_count"] > 410_000:
        raise RuntimeError("TinyResNet exceeds the frozen parameter budget")
    if check["architecture_macs"] > 105_000_000:
        raise RuntimeError("TinyResNet exceeds the frozen MAC budget")

print("GPU:", gender_e4_check["environment"]["gpu"])
print("Gender E1 parents:", gender_parent_run_ids)
print("Usage E2 parents: ", usage_parent_run_ids)
print("Gender E4:", gender_e4_check["model_family"], gender_e4_check["parameter_count"], "parameters")
print("Usage E4: ", usage_e4_check["model_family"], usage_e4_check["parameter_count"], "parameters")
print("MACs:", gender_e4_check["architecture_macs"], usage_e4_check["architecture_macs"])
print("Optimizer steps during checks:", gender_e4_check["optimizer_steps"], usage_e4_check["optimizer_steps"])


GPU: NVIDIA L4
Gender E1 parents: ('t3_baseline_gender_smallcnn_f0_s2753_e46cd00adf0a_20260830T082833Zf8c1e0', 't3_baseline_gender_smallcnn_f1_s2753_e46cd00adf0a_20260830T083708Z143950', 't3_baseline_gender_smallcnn_f2_s2753_e46cd00adf0a_20260830T084548Z5acbf0', 't3_baseline_gender_smallcnn_f3_s2753_e46cd00adf0a_20260830T085427Z5d34f9', 't3_baseline_gender_smallcnn_f4_s2753_e46cd00adf0a_20260830T090303Z41a843')
Usage E2 parents:  ('t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47')
Gender E4: task3_tinyresnet18_pm 394865 parameters
Usage E4:  task3_tinyresnet18_pm 395253 parameter

## 4. Train Gender E4: parameter-matched TinyResNet

This foreground cell trains folds 0–4. Only the model architecture changes from Gender E1;
ordinary cross-entropy and every training control remain fixed.


In [5]:
from fashion.train.task3_experiments import run_task3_child_cv

gender_e4_result = run_task3_child_cv(
    "gender_tinyresnet18_pm",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_e4_result


[task3] starting five-fold experiment=t3_gender_tinyresnet18_pm for target=gender
[task3] preparing target=gender fold=0: train=26,220, validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f0_s2753_561cdaebd672_20260831T024815Z283844; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.6120 train_macro_f1=0.4453 validation_loss=0.5962 validation_macro_f1=0.3588
[task3] target=gender fold=0 epoch=2/30 train_loss=0.4274 train_macro_f1=0.6119 validation_loss=0.5977 validation_macro_f1=0.5213
[task3] target=gender fold=0 epoch=3/30 train_loss=0.3680 train_macro_f1=0.6787 validation_loss=0.4733 validation_macro_f1=0.6424
[task3] target=gender fold=0 epoch=4/30 train_loss=0.3253 train_macro_f1=0.7123 validation_loss=0.4484 validation_macro_f1=0.6012
[task3] target=gender fold=0 epoch=5/30 train_loss

{'target': 'gender',
 'fold_run_ids': ['t3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f0_s2753_561cdaebd672_20260831T024815Z283844',
  't3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f1_s2753_561cdaebd672_20260831T025651Z0a7819',
  't3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f2_s2753_561cdaebd672_20260831T030526Z4d4fab',
  't3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f3_s2753_561cdaebd672_20260831T031408Zb77db2',
  't3_gender_e4_tinyresnet18_pm_gender_tinyresnet18pm_f4_s2753_561cdaebd672_20260831T032249Z808172'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e4_tinyresnet18_pm/gender/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e4_tinyresnet18_pm/gender/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e4_tinyresnet18_pm/gender/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/experiments/

## 5. Train Usage E4: parameter-matched TinyResNet

This foreground cell trains folds 0–4. It keeps the accepted E2 effective-number loss and
changes only SmallCNN to TinyResNet-18-PM.


In [6]:
usage_e4_result = run_task3_child_cv(
    "usage_tinyresnet18_pm",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_e4_result


[task3] starting five-fold experiment=t3_usage_tinyresnet18_pm for target=usage
[task3] preparing target=usage fold=0: train=26,219, validation=6,553
[task3] fitting fold-training RGB statistics for target=usage fold=0
[task3] RGB statistics ready for target=usage fold=0
[task3] registered t3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f0_s2753_f87812b0c178_20260831T033129Z7d4ec1; the first optimiser step may now run
[task3] target=usage fold=0 epoch=1/30 train_loss=1.2275 train_macro_f1=0.1797 validation_loss=1.6112 validation_macro_f1=0.1774
[task3] target=usage fold=0 epoch=2/30 train_loss=1.0832 train_macro_f1=0.2746 validation_loss=0.9224 validation_macro_f1=0.2777
[task3] target=usage fold=0 epoch=3/30 train_loss=0.8894 train_macro_f1=0.3271 validation_loss=0.9311 validation_macro_f1=0.3211
[task3] target=usage fold=0 epoch=4/30 train_loss=0.8603 train_macro_f1=0.3278 validation_loss=0.8255 validation_macro_f1=0.3300
[task3] target=usage fold=0 epoch=5/30 train_loss=0.7512 trai

{'target': 'usage',
 'fold_run_ids': ['t3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f0_s2753_f87812b0c178_20260831T033129Z7d4ec1',
  't3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f1_s2753_f87812b0c178_20260831T034010Z7f733c',
  't3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f2_s2753_f87812b0c178_20260831T034851Z173f05',
  't3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f3_s2753_f87812b0c178_20260831T035732Zc5da9c',
  't3_usage_e4_tinyresnet18_pm_usage_tinyresnet18pm_f4_s2753_f87812b0c178_20260831T040612Z397d10'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e4_tinyresnet18_pm/usage/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e4_tinyresnet18_pm/usage/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e4_tinyresnet18_pm/usage/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e4_tinyr

## 6. Show the first parent–child comparison

This is only a smoke check. The main Task 3 notebook must still apply every frozen primary,
paired-bootstrap, class, fold, calibration, robustness, and cost gate before accepting E4.


In [7]:
import pandas as pd

parent_metric_paths = {
    "gender": DRIVE_TASK_DIR / "baseline/gender/aggregate/metrics.json",
    "usage": DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/metrics.json",
}
children = {"gender": gender_e4_result, "usage": usage_e4_result}
comparison = []
for target, child in children.items():
    parent = json.loads(parent_metric_paths[target].read_text(encoding="utf-8"))
    comparison.append({
        "target": target,
        "parent_macro_f1": parent["macro_f1"],
        "e4_macro_f1": child["metrics"]["macro_f1"],
        "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
        "model_family": child["metrics"]["model_family"],
        "parameter_count": child["metrics"]["parameter_count"],
        "architecture_macs": child["metrics"]["architecture_macs"],
        "e4_metrics_path": child["metrics_path"],
    })
pd.DataFrame(comparison)


,target,parent_macro_f1,e4_macro_f1,macro_f1_change,model_family,parameter_count,architecture_macs,e4_metrics_path
0,gender,0.711753,0.712079,0.000326,task3_tinyresnet18_pm,394865,94268640,/content/drive/MyDrive/MLA2/task3/experiments/...
1,usage,0.408171,0.387820,-0.020350,task3_tinyresnet18_pm,395253,94269024,/content/drive/MyDrive/MLA2/task3/experiments/...
